In [ ]:
!pip install -q transformers
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, pipeline)
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt2_model.eval()
print("Both models are loaded. Let the showdown begin.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Both models are loaded. Let the showdown begin.


In [ ]:
bert_sentence = "The trophy didn't fit in the suitcase because [MASK] was too big."

print("BERT sees:", bert_sentence)
print()
bert_results = fill_mask(bert_sentence)
for r in bert_results[:5]:
    print(f"  {r['token_str']:>12s}")

BERT sees: The trophy didn't fit in the suitcase because [MASK] was too big.

            it
           she
            he
             i
          that


In [ ]:
gpt2_context = "The trophy didn't fit in the suitcase because"

print("GPT-2 sees only:", gpt2_context)
print()
inputs = gpt2_tokenizer(gpt2_context, return_tensors="pt")
with torch.no_grad():
    outputs = gpt2_model(**inputs)

# logits for the very next token position
next_token_logits = outputs.logits[0, -1]
probs = torch.softmax(next_token_logits, dim=-1)
top5 = torch.topk(probs, 5)

for score, idx in zip(top5.values, top5.indices):
    word = gpt2_tokenizer.decode(idx).strip()
    print(f"{word:>12s} confidence = {score.item():.3f}")

GPT-2 sees only: The trophy didn't fit in the suitcase because

          it confidence = 0.282
          of confidence = 0.121
         the confidence = 0.120
           I confidence = 0.119
          he confidence = 0.039


In [ ]:
# Just like our "Be the Attention Mechanism" classroom activity —
# watch what happens when we flip one single word.

for ending in ["big", "small"]:
    bert_sentence = f"The trophy didn't fit in the suitcase because [MASK] was too {ending}."
    print(f"=== ...because [MASK] was too {ending}. ===")
    for r in fill_mask(bert_sentence)[:3]:
        print(f"  {r['token_str']:>12}  confidence = {r['score']:.3f}")
    print()

=== ...because [MASK] was too big. ===
            it  confidence = 0.992
           she  confidence = 0.002
            he  confidence = 0.002

=== ...because [MASK] was too small. ===
            it  confidence = 0.995
           she  confidence = 0.002
            he  confidence = 0.001



In [ ]:
# TODO: write your own sentence and its left-half-only version
my_bert_sentence = "She poured the water into the glass until it was almost [MASK]."
my_gpt2_context = "She poured the water into the glass until it was almost"

print("--- BERT (both sides) ---")
for r in fill_mask(my_bert_sentence)[:3]:
    print(f"  {r['token_str']:>12s}   confidence = {r['score']:.3f}")

print()
print("--- GPT-2 (left side only) ---")
inputs = gpt2_tokenizer(my_gpt2_context, return_tensors="pt")
with torch.no_grad():
    outputs = gpt2_model(**inputs)
probs = torch.softmax(outputs.logits[0, -1], dim=-1)
top3 = torch.topk(probs, 3)

for score, idx in zip(top3.values, top3.indices):
    word = gpt2_tokenizer.decode(idx).strip()
    print(f"  {word:>12s}   confidence = {score.item():.3f}")

--- BERT (both sides) ---
         empty   confidence = 0.405
          full   confidence = 0.211
       boiling   confidence = 0.081

--- GPT-2 (left side only) ---
          full   confidence = 0.113
         empty   confidence = 0.074
         clear   confidence = 0.056


In [ ]:
!pip install -q transformers
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("zero-shot classifer is loaded,")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

zero-shot classifer is loaded,


In [ ]:
reviews = [
    "This movie was absolutely wonderful, I loved every minute of it!",
    "The plot was boring and the acting was terrible.",
    "The service was so good, I only waited forty five minutes.",
    "I don't dislike it, but I wouldn't watch it again either."
]

labels = ["positive", "negative", "neutral"]

for review in reviews:
    result = classifier(review, candidate_labels=labels)
    top_label = result["labels"][0]
    top_score = result["scores"][0]

    print(f"Review: {review}")
    print(f" -> {top_label} (confidence {top_score:.3f})")
    print()

Review: This movie was absolutely wonderful, I loved every minute of it!
 -> positive (confidence 0.981)

Review: The plot was boring and the acting was terrible.
 -> negative (confidence 0.984)

Review: The service was so good, I only waited forty five minutes.
 -> positive (confidence 0.963)

Review: I don't dislike it, but I wouldn't watch it again either.
 -> negative (confidence 0.631)



In [ ]:
# The real power of zero-shot classification: you can invent ANY labels you want,
# on the fly, with no retraining.

support_ticket = "my mobile screen turns blcak whenever i plug in charger."

topic_labels = ["billing", "hardware problem", "software bug", "account access"]

result = classifier(support_ticket, candidate_labels=topic_labels)

for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:>20}    {score:.3f}")

      hardware problem    0.684
        account access    0.123
               billing    0.109
          software bug    0.084


In [ ]:
# The real power of zero-shot classification: you can invent ANY labels you want,
# on the fly, with no retraining.

support_ticket = "I was charged twice for the same online order."

topic_labels = ["payment issue", "delivery problem", "account access", "product issue"]

result = classifier(support_ticket, candidate_labels=topic_labels)

for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:>20}    {score:.3f}")

         payment issue    0.677
      delivery problem    0.151
        account access    0.114
         product issue    0.057


In [ ]:
# TODO: write your own text and your own candidate labels

my_text = "I need this issue resolved before end of day, it's affecting my entire team."
my_labels = ["urgent", "not urgent", "spam"]

result = classifier(my_text, candidate_labels=my_labels)

for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:>15s}    {score:.3f}")

           urgent    0.992
       not urgent    0.005
             spam    0.003
